In [ ]:
!pip install llama-index llama-index-llms-google-genai llama-index-embeddings-google-genai torch sentence-transformers

INFO: pip is looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.3/303.3 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.3/107.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install llama-index-embeddings-huggingface

In [ ]:
import os
import nest_asyncio # <--- 1. Import the fix
nest_asyncio.apply() # <--- 2. Apply the fix

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.core.postprocessor import SimilarityPostprocessor, SentenceTransformerRerank
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# 1. SETUP: API Keys and Models
# ---------------------------------------------------------
# Replace with your actual Google Gemini API Key
os.environ["GOOGLE_API_KEY"] = "URKEY"

# Configure Gemini LLM
Settings.llm = GoogleGenAI(
    model="models/gemini-2.5-flash",
    temperature=0.1
)

# Configure Local Embeddings (Free, runs on Colab, no API key needed)
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

# Load your documents
if not os.path.exists("data"):
    os.makedirs("data")
    print("WARNING: Created 'data' folder. Please upload your document files into it before running!")
else:
    if not os.listdir("data"):
        print("WARNING: The 'data' folder is empty! Please upload your PDF/Text file into it.")
    else:
        documents = SimpleDirectoryReader("data").load_data()
        index = VectorStoreIndex.from_documents(documents)

# Define the Query to test
query = "What is the maximum loan amount a borrower can apply for?"

# 2. DEFINE THE EXPERIMENTS
# ---------------------------------------------------------
experiments = [
    {
        "name": "Experiment A (Default)",
        "top_k": 5,
        "threshold": None,
        "reranker": False,
        "notes": "Baseline"
    },
    {
        "name": "Experiment B (Filter)",
        "top_k": 8,
        "threshold": 0.50, # Threshold for local embeddings
        "reranker": False,
        "notes": "Filters weak matches"
    },
    {
        "name": "Experiment C (Rerank)",
        "top_k": 5,
        "threshold": 0.50,
        "reranker": True,
        "notes": "Adds LLM-based reranking"
    }
]

# Initialize the Reranker model
rerank_postprocessor = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-2-v2",
    top_n=3
)

# 3. RUN THE LOOP
# ---------------------------------------------------------
print(f"--- STARTING TASK: Run 3 RAG Configurations ---\n")
print(f"Query: {query}\n")

if 'index' in locals():
    for exp in experiments:
        print(f"==================================================")
        print(f"RUNNING: {exp['name']}")
        print(f"Config: top_k={exp['top_k']}, Threshold={exp['threshold']}, Reranker={exp['reranker']}")
        print(f"==================================================")

        # --- Step 1: Retrieval ---
        retriever = index.as_retriever(similarity_top_k=exp['top_k'])
        nodes = retriever.retrieve(query)

        # --- Step 2: Threshold Filtering ---
        if exp['threshold'] is not None:
            original_count = len(nodes)
            nodes = [node for node in nodes if node.score is not None and node.score > exp['threshold']]
            print(f"-> Threshold applied. Filtered from {original_count} to {len(nodes)} chunks.")

        # --- Step 3: Print Retrieved Chunks ---
        print("\n[RETRIEVED CHUNKS]:")
        if not nodes:
            print("No chunks passed the filter.")
        else:
            for i, node in enumerate(nodes):
                snippet = node.get_text().replace('\n', ' ')[:100]
                print(f"{i+1}. Score: {node.score:.4f} | Text: {snippet}...")

        # --- Step 4: Generation ---
        node_postprocessors = []
        if exp['reranker']:
            node_postprocessors.append(rerank_postprocessor)
        if exp['threshold']:
            node_postprocessors.append(SimilarityPostprocessor(similarity_cutoff=exp['threshold']))

        query_engine = index.as_query_engine(
            similarity_top_k=exp['top_k'],
            node_postprocessors=node_postprocessors
        )

        response = query_engine.query(query)

        # --- Step 5: Print Final Answer ---
        print(f"\n[FINAL GENERATED ANSWER]:\n{str(response)}\n")
        print("\n")
else:
    print("Error: Index not loaded. Please make sure you have documents in the 'data' folder.")

--- STARTING TASK: Run 3 RAG Configurations ---

Query: What is the maximum loan amount a borrower can apply for?

RUNNING: Experiment A (Default)
Config: top_k=5, Threshold=None, Reranker=False

[RETRIEVED CHUNKS]:
1. Score: 0.7045 | Text: Your actual rate, payment, and cost could be higher. Get an official Loan Estimate before choosing a...

[FINAL GENERATED ANSWER]:
The provided information does not specify the maximum loan amount a borrower can apply for. It only lists a "Total Loan Amount" of $380,000 for the example applicants, John Q. Smith and Mary A. Smith.



RUNNING: Experiment B (Filter)
Config: top_k=8, Threshold=0.5, Reranker=False
-> Threshold applied. Filtered from 1 to 1 chunks.

[RETRIEVED CHUNKS]:
1. Score: 0.7045 | Text: Your actual rate, payment, and cost could be higher. Get an official Loan Estimate before choosing a...

[FINAL GENERATED ANSWER]:
The provided information does not specify the maximum loan amount a borrower can apply for. It only details a specific